In [ ]:
# Notebook description / Structure

#this notebook is intended to analyze the broader economy as a whole and give a sense of the macroeconomic
#headwinds that will effect the asset and labor markets.

# The first section will be a brief overview of GDP which is the broadest measure of economic activity. and a deep analysis of gdp including
# the components of
# * GDP =
# * Consumption+
# * Investment+
# * Government Spending+
# * Net Exports (Exports - Imports)

In [ ]:
# TODOS
# plot the components of GDP which are
# 1. Consumption
# 2. Investment
# 3. Government Spending (current consumption and investment in capital goods)
# 4. Net Exports (Exports - Imports)

# get real gdp growht rate and nominal gdp growth rate
# get the gdp deflator

# work on visualzing and analyzing the
# the circular flow of income and expenditure
# three principal markets
# 1. goods and services market
# 2. labor market
# 3. (capital) / financial markets


In [ ]:
# Setup: Load libraries, parameters, and functions

from cmath import nan
from datetime import date
import copy
import sys
from pathlib import Path

# Make the repository-local Quantapp package importable regardless of the
# directory from which the Jupyter kernel was started.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "Quantapp" / "__init__.py").is_file():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not locate the Investment Research project root")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import matplotlib.pyplot as plt
from Quantapp.data import yf as qa_yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import datetime as dt
from plotly.subplots import make_subplots
import beaapi
from sklearn.preprocessing import MinMaxScaler
from Quantapp.data import MacroDataClient
from Quantapp.visualization import Plotter
from Quantapp.secrets import load_project_env, require_secret


plt.rcParams["figure.figsize"] = (10, 2)
load_project_env()
fred_api_key = require_secret("FRED_API_KEY")
beakey = require_secret("BEA_API_KEY")
ed = MacroDataClient(fred_key=fred_api_key)
qp = Plotter()

'''
list_of_sets = beaapi.get_data_set_list(beakey)
#Nipa = National Income and Product Accounts

list_of_parameters = beaapi.get_parameter_list(beakey,'NIPA')
list_of_param_vals = beaapi.get_parameter_values(beakey,'NIPA','TableID')
list_of_param_vals_frequency = beaapi.get_parameter_values(beakey, 'NIPA', 'Frequency')
display(list_of_sets)
display(list_of_parameters)
display(list_of_param_vals)
display(list_of_param_vals_frequency)
'''


def add_nber_recession_bands(fig, nber_series, fillcolor='LightGray', opacity=0.40):
    """Add every NBER recession interval to every subplot in a Plotly figure."""
    trace_dates = []
    for trace in fig.data:
        if getattr(trace, 'x', None) is None or len(trace.x) == 0:
            continue
        dates = pd.to_datetime(trace.x, errors='coerce', utc=True)
        dates = dates[~dates.isna()].tz_localize(None)
        if len(dates):
            trace_dates.extend((dates.min(), dates.max()))

    if not trace_dates:
        return fig

    figure_start, figure_end = min(trace_dates), max(trace_dates)
    recession = nber_series.copy()
    recession.index = pd.to_datetime(recession.index)
    recession = recession.sort_index().fillna(0).astype(float)

    # Include one observation before the visible range so an active recession
    # at the left boundary is not lost.
    prior = recession.loc[recession.index < figure_start].tail(1)
    visible = recession.loc[
        (recession.index >= figure_start) & (recession.index <= figure_end)
    ]
    recession = pd.concat([prior, visible])

    intervals = []
    in_recession = False
    recession_start = None
    for timestamp, value in recession.items():
        if value >= 0.5 and not in_recession:
            recession_start = max(timestamp, figure_start)
            in_recession = True
        elif value < 0.5 and in_recession:
            intervals.append((recession_start, min(timestamp, figure_end)))
            in_recession = False

    if in_recession and recession_start is not None:
        intervals.append((recession_start, figure_end))

    # Build all subplot rectangles at once. Repeated add_vrect calls force
    # Plotly to revalidate the entire growing figure for every interval.
    axis_numbers = sorted(
        int(name[5:] or 1) for name in fig.layout if name.startswith('xaxis')
    )
    shapes = list(fig.layout.shapes or ())
    for x0, x1 in intervals:
        for number in axis_numbers:
            suffix = '' if number == 1 else str(number)
            shapes.append(dict(
                type='rect', xref=f'x{suffix}', yref=f'y{suffix} domain',
                x0=x0, x1=x1, y0=0, y1=1, fillcolor=fillcolor,
                opacity=opacity, layer='below', line_width=0,
            ))
    fig.update_layout(shapes=shapes)
    return fig

def add_timeframe_dropdown(fig, default_years=50):
    """Add a consistent 50Y/30Y/20Y/10Y date-range dropdown to a Plotly figure."""
    trace_end_dates = []
    for trace in fig.data:
        if getattr(trace, 'x', None) is None or len(trace.x) == 0:
            continue
        dates = pd.to_datetime(trace.x, errors='coerce', utc=True)
        if not dates.isna().all():
            trace_end_dates.append(dates.max().tz_localize(None))

    if not trace_end_dates:
        return fig

    end_date = max(trace_end_dates)
    axis_names = sorted(
        (name for name in fig.layout if name.startswith('xaxis')),
        key=lambda name: int(name[5:] or 1),
    )
    timeframes = (50, 30, 20, 10, 5)
    buttons = []
    for years in timeframes:
        start_date = end_date - pd.DateOffset(years=years)
        relayout = {}
        for axis_name in axis_names:
            relayout[f'{axis_name}.range'] = [start_date, end_date]
            relayout[f'{axis_name}.autorange'] = False
        buttons.append(dict(label=f'{years}Y', method='relayout', args=[relayout]))

    fig.update_layout(
        updatemenus=[dict(
            type='dropdown',
            direction='down',
            active=timeframes.index(default_years),
            buttons=buttons,
            x=1.0,
            xanchor='right',
            y=1.13,
            yanchor='top',
            showactive=True,
        )],
        margin=dict(t=max(fig.layout.margin.t or 0, 100)),
    )
    default_start = end_date - pd.DateOffset(years=default_years)
    fig.update_xaxes(range=[default_start, end_date])
    return fig



## Economic framework

The U.S. economy can be viewed through six broad forces. Each force is monitored with a focused set of indicators.

### Consumers and households

| Area | Key indicators | Why it matters |
|---|---|---|
| Consumer activity | Retail Sales, Personal Income, Personal Spending, Consumer Confidence | Consumer spending accounts for roughly two-thirds of U.S. GDP. |
| Labor | Nonfarm Payrolls, Unemployment Rate, JOLTS, Weekly Jobless Claims, Wage Growth | Employment determines household income and spending power. |
| Housing | Housing Starts, Building Permits, Existing and New Home Sales | Housing is interest-rate-sensitive and often turns early in the business cycle. |
| Inflation | CPI, Core CPI, PCE, Core PCE, PPI | Inflation affects purchasing power and Federal Reserve policy. |

### Businesses

| Area | Key indicators | Why it matters |
|---|---|---|
| Business activity | ISM Manufacturing, ISM Services, Industrial Production, Durable Goods Orders, Capacity Utilization | Indicates whether firms are expanding or contracting. |
| Productivity | Labor Productivity, Unit Labor Costs | A long-run driver of living standards and corporate profitability. |

### Government

| Area | Key indicators | Why it matters |
|---|---|---|
| Fiscal activity | Government Spending, Federal Budget Balance, Treasury Issuance | Fiscal policy affects aggregate demand and financing conditions. |
| Economic output | GDP, Gross Domestic Income (GDI) | Broad measures of domestic economic activity and income. |

### Financial system

| Area | Key indicators | Why it matters |
|---|---|---|
| Credit | Bank Lending, Corporate Bond Spreads, Mortgage Rates, Delinquency Rates | Credit expansion supports growth; tightening can slow it. |
| Liquidity | Bank Reserves, M2 Money Supply, Overnight Reverse Repo Usage | Liquidity affects financial markets and funding availability. |

### Federal Reserve

| Area | Key indicators | Why it matters |
|---|---|---|
| Monetary policy | Federal Funds Rate, FOMC Statements, Dot Plot, Fed Balance Sheet | Determines the cost of money and influences system-wide liquidity. |
| Interest rates | 2-Year Treasury Yield, 10-Year Treasury Yield, 10-Year Real TIPS Yield, 10Y-2Y and 10Y-3M Yield Curves | Reflect policy, growth, inflation, and recession expectations. |

### Rest of the world

| Area | Key indicators | Why it matters |
|---|---|---|
| Trade | Trade Balance, Imports, Exports | Captures foreign demand, demand for foreign goods, and manufacturing exposure. |
| Capital flows and currency | U.S. Dollar Index (DXY), Cross-Border Capital Flows | Affect financial conditions, competitiveness, and dollar-sensitive assets. |

### Market-leading indicators

- High-yield credit spreads
- VIX
- U.S. Dollar Index (DXY)
- Treasury yields and yield curves
- Federal Reserve liquidity measures

These indicators often move before GDP because markets price expected future conditions rather than only current economic activity.


In [ ]:
# Setup: Download data

interest_rate_data = ed.get_interest_rate_data()
inflation_data = ed.get_inflation_data()
gdp_data = ed.get_gdp_data()
recession_data = ed.get_recession_indicators()

fred_key = fred_api_key
#Inflation data
#--------------------------------------------------
#Month over month growth rate
month_over_month_growth_rate_of_cpi = (inflation_data['CPI'].pct_change().mul(100).rename('CPI MoM % Change').dropna())
month_over_month_growth_rate_of_ppi = (inflation_data['PPI'].pct_change().mul(100).rename('PPI MoM % Change').dropna())
month_over_month_growth_rate_of_core_pce = (inflation_data['Core PCE'].pct_change().mul(100).rename('Core PCE MoM % Change').dropna())
month_over_month_growth_rate_of_core_cpi = (inflation_data['Core CPI'].pct_change().mul(100).rename('Core CPI MoM % Change').dropna())
month_over_month_growth_rate_of_core_ppi = (inflation_data['Core PPI'].pct_change().mul(100).rename('Core PPI MoM % Change').dropna())

#3 month annualized growth rate
annualized_3m_growth_rate_of_cpi = (( (inflation_data['CPI'] / inflation_data['CPI'].shift(3)) ** 4 - 1 ).mul(100).rename('CPI 3-Month Annualized % Change').dropna())
annualized_3m_growth_rate_of_ppi = (( (inflation_data['PPI'] / inflation_data['PPI'].shift(3)) ** 4 - 1 ).mul(100).rename('PPI 3-Month Annualized % Change').dropna())
annualized_3m_growth_rate_of_core_pce = (( (inflation_data['Core PCE'] / inflation_data['Core PCE'].shift(3)) ** 4 - 1 ).mul(100).rename('Core PCE 3-Month Annualized % Change').dropna())
annualized_3m_growth_rate_of_core_cpi = (( (inflation_data['Core CPI'] / inflation_data['Core CPI'].shift(3)) ** 4 - 1 ).mul(100).rename('Core CPI 3-Month Annualized % Change').dropna())
annualized_3m_growth_rate_of_core_ppi = (( (inflation_data['Core PPI'] / inflation_data['Core PPI'].shift(3)) ** 4 - 1 ).mul(100).rename('Core PPI 3-Month Annualized % Change').dropna())


#6 month annualized growth rate
annualized_6m_growth_rate_of_cpi = (( (inflation_data['CPI'] / inflation_data['CPI'].shift(6)) ** 2 - 1 ).mul(100).rename('CPI 6-Month Annualized % Change').dropna())
annualized_6m_growth_rate_of_ppi = (( (inflation_data['PPI'] / inflation_data['PPI'].shift(6)) ** 2 - 1 ).mul(100).rename('PPI 6-Month Annualized % Change').dropna())
annualized_6m_growth_rate_of_core_pce = (( (inflation_data['Core PCE'] / inflation_data['Core PCE'].shift(6)) ** 2 - 1 ).mul(100).rename('Core PCE 6-Month Annualized % Change').dropna())
annualized_6m_growth_rate_of_core_cpi = (( (inflation_data['Core CPI'] / inflation_data['Core CPI'].shift(6)) ** 2 - 1 ).mul(100).rename('Core CPI 6-Month Annualized % Change').dropna())
annualized_6m_growth_rate_of_core_ppi = (( (inflation_data['Core PPI'] / inflation_data['Core PPI'].shift(6)) ** 2 - 1 ).mul(100).rename('Core PPI 6-Month Annualized % Change').dropna())

#12 month growth rate
annual_growth_rate_of_cpi = (inflation_data['CPI'].pct_change(12).mul(100).rename('CPI 12-Month % Change').dropna())
annual_growth_rate_of_ppi = (inflation_data['PPI'].pct_change(12).mul(100).rename('PPI 12-Month % Change').dropna())
annual_growth_rate_of_core_pce = (inflation_data['Core PCE'].pct_change(12).mul(100).rename('Core PCE 12-Month % Change').dropna())
annual_growth_rate_of_core_cpi = (inflation_data['Core CPI'].pct_change(12).mul(100).rename('Core CPI 12-Month % Change').dropna())
annual_growth_rate_of_core_ppi = (inflation_data['Core PPI'].pct_change(12).mul(100).rename('Core PPI 12-Month % Change').dropna())
#--------------------------------------------------

#Categories of inflation
#--------------------------------------------------
#cpi_commodities = fetch_fred_json(cpi_commodities_query).rename(columns={'value': 'CPI Commodities'})
#cpi_energy = fetch_fred_json(cpi_energy_query).rename(columns={'value': 'CPI Energy'})
#cpi_food = fetch_fred_json(cpi_food_query).rename(columns={'value': 'CPI Food'})
#cpi_services = fetch_fred_json(cpi_services_query).rename(columns={'value': 'CPI Services'})
#cpi_shelter = fetch_fred_json(cpi_shelter_query).rename(columns={'value': 'CPI Shelter'})
#cpi_rent_of_primary_residence = fetch_fred_json(cpi_rent_of_primary_residence_query).rename(columns={'value': 'CPI Rent of Primary Residence'})
#cpi_owners_equivalent_rent = fetch_fred_json(cpi_owners_equivalent_rent_query).rename(columns={'value': 'CPI Owners Equivalent Rent'})

#do ppi and pce categories later!!!!!
#--------------------------------------------------

#interest Rates
#--------------------------------------------------
fed_funds_rate = interest_rate_data['Federal Funds Rate']
treasury_10yr_rate = interest_rate_data['10-Year Treasury Constant Maturity Rate']
tips_10yr_rate = interest_rate_data['10-Year TIPS Rate']
aaa_corporate_bond_yield = interest_rate_data['AAA Corporate Bond Yield']
#--------------------------------------------------


#real interest rates
#--------------------------------------------------
real_fed_funds_rate = (interest_rate_data['Federal Funds Rate'] - annual_growth_rate_of_core_pce).dropna()
real_10yr_treasury_rate = (interest_rate_data['10-Year Treasury Constant Maturity Rate'] - annual_growth_rate_of_core_pce).dropna()
real_10yr_tips_rate = (interest_rate_data['10-Year TIPS Rate'] - annual_growth_rate_of_core_pce).dropna()
breakeven_inflation_10yr = (interest_rate_data['10-Year Treasury Constant Maturity Rate'] - interest_rate_data['10-Year TIPS Rate']).dropna()
real_aaa_corporate_bond_yield = (interest_rate_data['AAA Corporate Bond Yield'] - annual_growth_rate_of_core_pce).dropna()
real_baa_corporate_bond_yield = (interest_rate_data['BAA Corporate Bond Yield'] - annual_growth_rate_of_core_pce).dropna()
#--------------------------------------------------


#recession data
#--------------------------------------------------
nber = recession_data['NBER Recession Indicators']
oecd = recession_data['OECD Recession Indicators']
real_time_sahm_rule = recession_data['Real-Time Sahm Rule']
markov_switching_smoothed = recession_data['Markov Switching Smoothed Probability']
#--------------------------------------------------

#Risk assets data
#--------------------------------------------------
risk_asset_names = {
    '^GSPC': 'S&P 500 Index', 'DBC': 'DBC Commodity Index',
    'VNQ': 'VNQ Real Estate Index', 'GLD': 'GLD Gold ETF',
    'USO': 'USO Oil ETF', 'AGG': 'AGG Bond ETF',
    'LQD': 'LQD Investment Grade Corporate Bond ETF',
    'HYG': 'HYG High Yield Corporate Bond ETF',
}
risk_asset_history = qa_yf.download(
    list(risk_asset_names), period='max', auto_adjust=True,
    progress=False, threads=True,
)
risk_assets_df = risk_asset_history['Close'].rename(columns=risk_asset_names).dropna()

risk_assets_3_month_percent_change = np.log(risk_assets_df).diff(3).dropna().rename(columns=lambda x: f"{x} 3-Month Log Return")
risk_assets_6_month_percent_change = np.log(risk_assets_df).diff(6).dropna().rename(columns=lambda x: f"{x} 6-Month Log Return")
risk_assets_12_month_percent_change = np.log(risk_assets_df).diff(12).dropna().rename(columns=lambda x: f"{x} 12-Month Log Return")


#--------------------------------------------------





#create a plotly function that will plot the data
def plotly_line(df, title, x_label, y_label):
    import plotly.express as px
    fig = px.line(df, title=title, labels={df.index.name: x_label, 'value': y_label})
    return fig

#print(unemployment_rate_df)

In [ ]:
# 1. Labor Market: Can consumers earn money?
# Labor is the engine of consumer spending. If jobs weaken, downstream activity usually weakens.

labor_series_ids = {
    'Nonfarm Payrolls': 'PAYEMS',
    'Unemployment Rate': 'UNRATE',
    'Weekly Initial Jobless Claims': 'ICSA',
    'JOLTS Job Openings': 'JTSJOL',
    'Average Hourly Earnings': 'CES0500000003',
}

labor_market_data = ed.fetch_fred_series(labor_series_ids)

labor_plot_config = [
    ('Nonfarm Payrolls', 'YoY change (%)', 'yoy'),
    ('Unemployment Rate', 'Percent', 'level'),
    ('Weekly Initial Jobless Claims', 'Claims, 4-week average', 'four_week_average'),
    ('JOLTS Job Openings', 'Thousands of openings', 'level'),
    ('Average Hourly Earnings', 'YoY change (%)', 'yoy'),
]

fig = make_subplots(
    rows=len(labor_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.035,
    subplot_titles=[
        (f'{name} - YoY %' if view == 'yoy' else
         f'{name} - 4-Week Moving Average' if view == 'four_week_average' else name)
        for name, _, view in labor_plot_config
    ],
)

for row, (name, unit, view) in enumerate(labor_plot_config, start=1):
    raw_series = labor_market_data[name].dropna()
    if view == 'yoy':
        series = raw_series.pct_change(12).mul(100).dropna()
    elif view == 'four_week_average':
        series = raw_series.rolling(4).mean().dropna()
    else:
        series = raw_series
    display_name = (
        f'{name} - YoY %' if view == 'yoy' else
        f'{name} - 4-Week Moving Average' if view == 'four_week_average' else name
    )
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=display_name,
            mode='lines',
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>' + display_name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Labor Market Indicators',
    template='plotly_dark',
    height=1_150,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=90, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(labor_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()

In [ ]:
# 2. Consumer: Are people spending?
# Consumers account for roughly 70% of U.S. GDP, making income and spending key growth signals.

consumer_series_ids = {
    'Personal Income': 'PI',
    'Retail Sales': 'RSAFS',
    'Personal Consumption Expenditures (PCE)': 'PCE',
}

consumer_data = ed.fetch_fred_series(consumer_series_ids)

consumer_rate_views = {}
for metric in consumer_series_ids:
    level = consumer_data[metric].dropna()
    consumer_rate_views[metric] = pd.DataFrame({
        'YoY %': level.pct_change(12).mul(100),
    }).dropna(how='all')

consumer_metrics = list(consumer_series_ids)

fig = make_subplots(
    rows=len(consumer_metrics),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f'{metric} - YoY %' for metric in consumer_metrics],
)

for row, metric in enumerate(consumer_metrics, start=1):
    series = consumer_rate_views[metric]['YoY %'].dropna()
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode='lines',
            name=f'{metric} - YoY %',
            line=dict(color='green'),
            showlegend=False,
            hovertemplate=(
                f'{metric} - YoY %<br>'
                '%{x|%Y-%m}<br>%{y:.2f}%<extra></extra>'
            ),
        ),
        row=row,
        col=1,
    )
    fig.add_hline(y=0, line_color='gray', line_width=1, row=row, col=1)
    fig.update_yaxes(title_text='Percent', row=row, col=1)

fig.update_xaxes(title_text='Date', row=len(consumer_metrics), col=1)

fig.update_layout(
    title='U.S. Consumer Income and Spending Growth',
    template='plotly_dark',
    height=1_050,
    hovermode='x unified',
    margin=dict(l=90, r=90, t=120, b=60),
)

add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 3. Inflation: Are prices rising?
breakeven_10yr_rate = pd.DataFrame(
    {
       #'10-Year Breakeven Inflation Rate': treasury_10yr_rate['10-Year Treasury Rate'] - tips_10yr_rate['10-Year TIPS Rate']
         '10-Year Breakeven Inflation Rate': interest_rate_data['10-Year Treasury Constant Maturity Rate'] - interest_rate_data['10-Year TIPS Rate']
    }
).dropna()

breakeven_monthly = breakeven_10yr_rate['10-Year Breakeven Inflation Rate'].resample('MS').mean()
breakeven_aligned, pce_aligned = breakeven_monthly.align(annual_growth_rate_of_core_pce, join='inner')
breakeven_pce_spread = (breakeven_aligned - pce_aligned).rename("10-Year Breakeven - Core PCE Spread")
'''
recession_index = annual_growth_rate_of_core_pce.index.union(
    annual_growth_rate_of_core_cpi.index
).union(
    annual_growth_rate_of_core_ppi.index
).union(
    current_real_rates.index
).union(
    breakeven_10yr_rate.index
).union(
    breakeven_pce_spread.index
)'''

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        "10-Year Breakeven & Core Inflation Measures",
        "Real Interest Rates"
    ]
)
'''
fig.add_trace(go.Scatter(
    x=breakeven_10yr_rate.index,
    y=breakeven_10yr_rate['10-Year Breakeven Inflation Rate'],
    mode='lines',
    name='10-Year Breakeven Inflation',
    line=dict(color='yellow')
), row=1, col=1)
'''
fig.add_trace(go.Scatter(
    x=annual_growth_rate_of_core_pce.index,
    y=annual_growth_rate_of_core_pce,
    mode='lines',
    name='Core PCE 12-Month % Change',
    line=dict(color='blue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=annual_growth_rate_of_core_cpi.index,
    y=annual_growth_rate_of_core_cpi,
    mode='lines',
    name='Core CPI 12-Month % Change',
    line=dict(color='orange')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=annual_growth_rate_of_core_ppi.index,
    y=annual_growth_rate_of_core_ppi,
    mode='lines',
    name='Core PPI 12-Month % Change',
    line=dict(color='green')
), row=1, col=1)

fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="red",
    annotation_text="2% Target",
    annotation_position="top left",
    row=1,
    col=1
)

fig.add_trace(go.Scatter(
    x=real_fed_funds_rate.index,
    y=real_fed_funds_rate,
    mode='lines',
    name='Real Fed Funds Rate',
    line=dict(color='blue')
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=real_10yr_treasury_rate.index,
    y=real_10yr_treasury_rate,
    mode='lines',
    name='Real 10-Year Treasury Rate',
    line=dict(color='orange')
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=real_10yr_tips_rate.index,
    y=real_10yr_tips_rate,
    mode='lines',
    name='Real 10-Year TIPS Rate',
    line=dict(color='green')
), row=2, col=1)

#add_recession_bands(fig, nber_df=nber_df, index=recession_index)
#add_recession_bands(fig, nber_df=nber_df, index=fed_funds_rate.index)
#add_recession_bands(fig, nber_df=nber_df, index=real_fed_funds_rate.index)
fig.update_yaxes(title_text="Inflation Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Real Rate (%)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

inflation_end_date = max(
    annual_growth_rate_of_core_pce.index.max(),
    annual_growth_rate_of_core_cpi.index.max(),
    annual_growth_rate_of_core_ppi.index.max(),
    real_fed_funds_rate.index.max(),
)
inflation_timeframes = {
    '50Y': inflation_end_date - pd.DateOffset(years=50),
    '30Y': inflation_end_date - pd.DateOffset(years=30),
    '20Y': inflation_end_date - pd.DateOffset(years=20),
    '10Y': inflation_end_date - pd.DateOffset(years=10),
}

timeframe_buttons = [
    dict(
        label=label,
        method='relayout',
        args=[{
            'xaxis.range': [start_date, inflation_end_date],
            'xaxis2.range': [start_date, inflation_end_date],
        }],
    )
    for label, start_date in inflation_timeframes.items()
]

fig.update_layout(
    title="Core Inflation Measures and Real Rates with NBER Recessions",
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(x=0, y=1),
    height=900,
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        active=0,
        buttons=timeframe_buttons,
        x=1.0,
        xanchor='right',
        y=1.13,
        yanchor='top',
        showactive=True,
    )],
    annotations=list(fig.layout.annotations) + [dict(
        text='Time frame:',
        x=0.86,
        xref='paper',
        y=1.105,
        yref='paper',
        showarrow=False,
    )],
)
fig.update_xaxes(
    range=[inflation_timeframes['50Y'], inflation_end_date],
)

fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='LightPink', row=2, col=1)

add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 3A. Inflation: Year-over-year change

inflation_measure_columns = {
    'CPI': 'CPI',
    'PPI': 'PPI',
    'Core PCE': 'Core PCE',
    'Core CPI': 'Core CPI',
    'Core PPI': 'Core PPI',
}

fig = make_subplots(
    rows=len(inflation_measure_columns),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[f'{name} - YoY %' for name in inflation_measure_columns],
)

for row, (display_name, column) in enumerate(inflation_measure_columns.items(), start=1):
    series = inflation_data[column].pct_change(12).mul(100).dropna()
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode='lines',
            name=f'{display_name} - YoY %',
            showlegend=False,
            hovertemplate=(
                f'{display_name} YoY<br>'
                '%{x|%Y-%m}<br>%{y:.2f}%<extra></extra>'
            ),
        ),
        row=row,
        col=1,
    )
    fig.add_hline(y=0, line_color='gray', line_width=1, row=row, col=1)
    fig.update_yaxes(title_text='YoY change (%)', row=row, col=1)

fig.update_xaxes(title_text='Date', row=len(inflation_measure_columns), col=1)
fig.update_layout(
    title='U.S. Inflation Measures: Year-over-Year Change',
    template='plotly_dark',
    hovermode='x unified',
    height=1500,
)

add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 4. Business Activity: Are companies expanding?
# These indicators show whether businesses are growing, hiring, and investing.
# Industrial Production is a core measure of real economic activity.

business_fred_series_ids = {
    'Industrial Production': 'INDPRO',
    'Durable Goods Orders': 'DGORDER',
    'Capacity Utilization': 'TCU',
}
business_fred_data = ed.fetch_fred_series(business_fred_series_ids)

business_activity_data = business_fred_data

business_plot_config = [
    ('Industrial Production', 'YoY change (%)', 'yoy'),
    ('Durable Goods Orders', 'Millions of dollars', 'level'),
    ('Capacity Utilization', 'Percent', 'level'),
]

fig = make_subplots(
    rows=len(business_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.035,
    subplot_titles=[f'{name} - YoY %' if view == 'yoy' else name for name, _, view in business_plot_config],
)

for row, (name, unit, view) in enumerate(business_plot_config, start=1):
    raw_series = business_activity_data[name].dropna()
    series = raw_series.pct_change(12).mul(100).dropna() if view == 'yoy' else raw_series
    display_name = f'{name} - YoY %' if view == 'yoy' else name
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=display_name,
            mode='lines',
            hovertemplate='%{x|%Y-%m}<br>%{y:,.2f}<extra>' + display_name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Business Activity Indicators',
    template='plotly_dark',
    height=850,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=100, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(business_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()

In [ ]:
# 5. Housing
# Housing is one of the most interest-rate-sensitive sectors.

housing_series_ids = {
    'Housing Starts': 'HOUST',
    'Building Permits': 'PERMIT',
    'Existing Home Sales': 'EXHOSLUSM495S',
    'New Home Sales': 'HSN1F',
}

housing_data = ed.fetch_fred_series(housing_series_ids)

housing_plot_config = [
    ('Housing Starts', 'YoY change (%)', 'yoy'),
    ('Building Permits', 'Thousands of units, SAAR', 'level'),
    ('Existing Home Sales', 'Thousands, SAAR', 'level'),
    ('New Home Sales', 'Thousands of units, SAAR', 'level'),
]

fig = make_subplots(
    rows=len(housing_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[f'{name} - YoY %' if view == 'yoy' else name for name, _, view in housing_plot_config],
)

for row, (name, unit, view) in enumerate(housing_plot_config, start=1):
    raw_series = housing_data[name].dropna()
    series = raw_series.pct_change(12).mul(100).dropna() if view == 'yoy' else raw_series
    display_name = f'{name} - YoY %' if view == 'yoy' else name
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=display_name,
            mode='lines',
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>' + display_name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Housing Indicators',
    template='plotly_dark',
    height=950,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=105, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(housing_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 6. Monetary Policy
# The Federal Reserve controls the price and availability of money.

monetary_policy_series_ids = {
    'Fed Funds Rate (Monthly)': 'FEDFUNDS',
    'Fed Balance Sheet: Total Assets': 'WALCL',
    'Effective Fed Funds Rate (Daily)': 'DFF',
}

monetary_policy_data = ed.fetch_fred_series(monetary_policy_series_ids)

monetary_policy_plot_config = [
    ('Fed Funds Rate (Monthly)', 'Percent'),
    ('Fed Balance Sheet: Total Assets', 'Millions of dollars'),
    ('Effective Fed Funds Rate (Daily)', 'Percent'),
]

fig = make_subplots(
    rows=len(monetary_policy_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[name for name, _ in monetary_policy_plot_config],
)

for row, (name, unit) in enumerate(monetary_policy_plot_config, start=1):
    series = monetary_policy_data[name].dropna()
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=name,
            mode='lines',
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>' + name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Monetary Policy Indicators',
    template='plotly_dark',
    height=850,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=105, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(monetary_policy_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 7. Interest Rates
# Bond markets often react before the broader economy.

interest_rates_series_ids = {
    '2-Year Treasury': 'DGS2',
    '10-Year Treasury': 'DGS10',
    '10-Year TIPS Yield': 'DFII10',
    '10Y-2Y Spread': 'T10Y2Y',
    '10Y-3M Spread': 'T10Y3M',
}

interest_rates_data = ed.fetch_fred_series(interest_rates_series_ids)

interest_rates_plot_config = [
    ('2-Year Treasury', 'Percent'),
    ('10-Year Treasury', 'Percent'),
    ('10-Year TIPS Yield', 'Percent'),
    ('10Y-2Y Spread', 'Percentage points'),
    ('10Y-3M Spread', 'Percentage points'),
]

fig = make_subplots(
    rows=len(interest_rates_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[name for name, _ in interest_rates_plot_config],
)

for row, (name, unit) in enumerate(interest_rates_plot_config, start=1):
    series = interest_rates_data[name].dropna()
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=name,
            mode='lines',
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>' + name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Interest Rates Indicators',
    template='plotly_dark',
    height=1150,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=105, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(interest_rates_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 8. Credit
# Credit conditions often tighten before recessions.

credit_series_ids = {
    'High-Yield Spread': 'BAMLH0A0HYM2',
    'Investment-Grade Spread': 'BAMLC0A0CM',
    '30-Year Mortgage Rate': 'MORTGAGE30US',
    'Bank Lending': 'TOTLL',
}

credit_data = ed.fetch_fred_series(credit_series_ids)

credit_plot_config = [
    ('High-Yield Spread', 'Percentage points'),
    ('Investment-Grade Spread', 'Percentage points'),
    ('30-Year Mortgage Rate', 'Percent'),
    ('Bank Lending', 'Billions of dollars'),
]

fig = make_subplots(
    rows=len(credit_plot_config),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[name for name, _ in credit_plot_config],
)

for row, (name, unit) in enumerate(credit_plot_config, start=1):
    series = credit_data[name].dropna()
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            name=name,
            mode='lines',
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>' + name + '</extra>',
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=unit, row=row, col=1)

fig.update_layout(
    title='U.S. Credit Indicators',
    template='plotly_dark',
    height=950,
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=105, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Date', row=len(credit_plot_config), col=1)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()


In [ ]:
# 9. GDP and economic output
gdp_data_all = ed.get_gdp_data()
gdp_data = gdp_data_all[['Nominal GDP', 'Real GDP']].dropna()
gdp_data_components = gdp_data_all[['Real GDP', 'Consumption Expenditures', 'Investment', 'Government Spending', 'Exports', 'Imports', 'Net Exports']].dropna()
gdp_data_components_real = gdp_data_all[['Real GDP', 'Real Consumption Expenditures', 'Real Investment', 'Real Government Spending', 'Real Exports', 'Real Imports', 'Real Net Exports']].dropna()
#display(gdp_data_components)
#plot Nominal GDP
fig = px.line(
    gdp_data[['Nominal GDP']],
    title="Nominal GDP Over Time",
    labels={"index": "Date", "value": "GDP (in Billions of Dollars)", "variable": "GDP Type"}
)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()

gdp_data_components

#plot components of GDP
fig = px.line(
    gdp_data_components,
    title="Components of GDP Over Time",
    labels={"index": "Date", "value": "GDP Components (in Billions of Dollars)", "variable": "GDP Component"}
)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()

#plot Real GDP
fig = px.line(
    gdp_data[['Real GDP']],
    title="Real GDP Over Time",
    labels={"index": "Date", "value": "Real GDP (in Billions of Dollars)", "variable": "GDP Type"}
)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()

#plot components of Real GDP
fig = px.line(
    gdp_data_components_real,
    title="Components of Real GDP Over Time",
    labels={"index": "Date", "value": "Real GDP Components (in Billions of Dollars)", "variable": "Real GDP Component"}
)
add_nber_recession_bands(fig, nber)
add_timeframe_dropdown(fig)
fig.show()
